In [1]:
%pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

from fastai.vision.all import *
from fastbook import *

matplotlib.rc('image', cmap='Greys')

Note: you may need to restart the kernel to use updated packages.


# The forward pass with pixel similarity (Prediction)

In [11]:
# Training Set
path = untar_data(URLs.MNIST_SAMPLE)

Path.BASE_PATH = path
path.ls()

(path/'train').ls()

threes = (path/'train'/'3').ls().sorted()
sevens = (path/'train'/'7').ls().sorted()


# Load training set into tensors
seven_tensors = [tensor(Image.open(o)) for o in sevens]
three_tensors = [tensor(Image.open(o)) for o in threes]
len(three_tensors),len(seven_tensors)

# stack the tensors (N, 28,28) where N is number of images
# stacking tensors allows for fast calculations to run on all the images at once
stacked_sevens = torch.stack(seven_tensors).float()/255
stacked_threes = torch.stack(three_tensors).float()/255
stacked_threes.shape

# Get the Mean using the stacked tensors this creates our 'perfect' 3 and 7 
mean3 = stacked_threes.mean(0)
mean7 = stacked_sevens.mean(0)

# To find the distance of one image to our 'perfect' 3 or 7 Use L1 or MSE (i use MSE for this example)
a_3 = stacked_threes[1]
dist_3_sqr = ((a_3 - mean3)**2).mean().sqrt()
dist_7_sqr = ((a_3 - mean7)**2).mean().sqrt()
print(f"distance from 3: {dist_3_sqr}" , f"distance from 7: {dist_7_sqr} ")


# Pytorches loss functions
print(f"Pytorch l1 on mean3: {F.l1_loss(a_3.float(), mean3)}")
print(f"Pythorch mse on mean7: {F.mse_loss(a_3.float(), mean7)}")


# Validation set

valid_3_tens = torch.stack([tensor(Image.open(o)) for o in (path/'valid'/'3').ls()])
valid_3_tens = valid_3_tens.float()/255

valid_7_tens = torch.stack([tensor(Image.open(o)) for o in (path/'valid'/'7').ls()])
valid_7_tens = valid_7_tens.float()/255


# Loss function using distance
# mean((-1, -2)) is a pytorch to mean the last -1 and second last -2 dimensions e.g. [1010,28,28] the actual image pixels are mean
# this is flexible for all 2d images because the last two dimeinsion represent the image
# a-b allows pytorch to use broadcasting where the smaller tensor will have a virtal matching tensor so that the tensor arithmetic can happen
# because we mean((-1,-2)) the new dimensions added on do affect the desired result, ensures the calculation itself stays focused only on pixels.
def mnist_distance(a,b): return (a-b).abs().mean((-1,-2))
mnist_distance(a_3, mean3)


# Linear classification is it a 3 or 7 this is used to check if our prediction is correct
def is_3(x): return mnist_distance(x,mean3) < mnist_distance(x,mean7)
print(f"is a_3 a three? {is_3(a_3).float()}")

# test our model
accuracy_3s = is_3(valid_3_tens).float() .mean()
accuracy_7s = (1 - is_3(valid_7_tens).float()).mean()
print(f"models performance on 3s {accuracy_3s}", f"models performance on 7s {accuracy_7s}")


distance from 3: 0.20208320021629333 distance from 7: 0.30210891366004944 
Pytorch l1 on mean3: 0.11143654584884644
Pythorch mse on mean7: 0.09126979112625122
is a_3 a three? 1.0
models performance on 3s 0.9168316721916199 models performance on 7s 0.9854085445404053


# The backwards pass (learning) with SDG
1. *Initialize* the weights.
1. For each image, use these weights to *predict* whether it appears to be a 3 or a 7.
1. Based on these predictions, calculate how good the model is (its *loss*).
1. Calculate the *gradient*, which measures for each weight, how changing that weight would change the loss
1. *Step* (that is, change) all the weights based on that calculation.
1. Go back to the step 2, and *repeat* the process.
1. Iterate until you decide to *stop* the training process (for instance, because the model is good enough or you don't want to wait any longer).

# Prepare the data
Training set with labels
Validation set with labels

In [12]:
# training set
# View changes the shape of the data into something a neural network can use.
# torch cat will stack two tensors together. the -1 means that pytorch should figure out what to do with it.
# the view function changes the shape of a tensor without changing its data
# stack_threes and stack_sevens are tensors of the same size, but the dimension we stack on (batch) doesn't need to be the same.

# torch.cat combines the data set into one along the batch size dimension [b, h, w]
# view then changes the shape of the tensor to 2d tensor [total_images, h, w] => [total_images, hxw] each row is a flatten image, each column one position across all the images

# x is the training set for 3s and 7s, they are all ordered with 3s first then 7s (you can use 3s.length to get the start of 7s)
train_x = torch.cat([stacked_threes, stacked_sevens]).view(-1, 28*28)
# y is the labels on the training set
train_y = tensor([1]*len(threes) + [0]*len(sevens)).unsqueeze(1)

# pair the training set and labels into a dataset
dset = list(zip(train_x,train_y))
x,y = dset[0]

# Validation set
# Zip pairs the images with their labels
valid_x = torch.cat([valid_3_tens, valid_7_tens]).view(-1, 28*28)
valid_y = tensor([1]*len(valid_3_tens) + [0]*len(valid_7_tens)).unsqueeze(1)
valid_dset = list(zip(valid_x,valid_y))

In [15]:
# Step 1: Initialize the weights and bias randomly
def init_params(size, std=1.0): return (torch.randn(size)*std).requires_grad_()
bias = init_params(1)
weights = init_params((28*28,1)) # because of the dot product weghts needs 784 rows and 1 column [784,1] to match with [784,1] from train_x

# Step 2: make a prediction, sum is required for the dot product.
(train_x[0]*weights.T).sum() + bias

# Step 3: Make many predictions use matrix multiplication
def linear1(xb): return xb@weights + bias
preds = linear1(train_x)
preds

# Step 4: Calculate the loss
# (preds > 0.0) → Creates a boolean tensor: True if logit is positive, False otherwise
# .float() → Converts booleans to 1.0 and 0.0 so it matches the shape/dtype of train_y
# == train_y → Element-wise comparison: True where prediction matches true label
# .float().mean().item() → Computes accuracy (fraction of correct predictions)
corrects = (preds>0.0).float() == train_y
print(f" correct guesses {corrects}")
print(f" prediction score {corrects.float().mean().item()}")

# Step 4.1 Loss function with sigmoid
# Use the sigmoid function to make sure our predictions are between 0 and 1.
def mnist_loss(predictions, targets):
    predictions = predictions.sigmoid()
    return torch.where(targets==1, 1-predictions, predictions).mean()

# Step 4.1 Create batches of data to prevent overfitting
weights = init_params((28*28,1))
bias = init_params(1) # bias used to tilt answer towards an outcome and is important for our model to learn.

# randomised mini batches of data from the training set.
dl = DataLoader(dset, batch_size=256) # get 256 random images from dset
xb,yb = first(dl) # xb is the data, yb is the labels
print(xb.shape,yb.shape)

# randomise mini batches of data for the training set.
valid_dl = DataLoader(valid_dset, batch_size=256) # get 256 random images from valid_dset

# Accuracy not the loss function
def batch_accuracy(xb, yb):
    preds = xb.sigmoid()
    correct = (preds>0.5) == yb
    return correct.float().mean()

# gradiant function
def calc_grad(xb, yb, model):
    preds = model(xb)
    loss = mnist_loss(preds, yb)
    loss.backward()

# Imporant to not let pytorch also track the step (where we update the weights) as it will be tracked and added to the computational graph.
# This is why we need to use .data
def train_epoch(model, lr, params):
    for xb,yb in dl:
        calc_grad(xb, yb, model)
        for p in params:
            p.data -= p.grad*lr
            p.grad.zero_()


def validate_epoch(model):
    accs = [batch_accuracy(model(xb), yb) for xb,yb in valid_dl]
    return round(torch.stack(accs).mean().item(), 4)

# train
lr = 1.
params = weights,bias
for i in range(20):
    train_epoch(linear1, lr, params)
    print(validate_epoch(linear1), end=' ')



 correct guesses tensor([[False],
        [False],
        [ True],
        ...,
        [ True],
        [ True],
        [ True]])
 prediction score 0.6337528228759766
torch.Size([256, 784]) torch.Size([256, 1])
0.7016 0.8646 0.9178 0.9398 0.9481 0.9535 0.9574 0.9593 0.9622 0.9642 0.9661 0.9686 0.9691 0.9701 0.9701 0.972 0.973 0.973 0.9735 0.974 